In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [3]:
# ================================
# 🌲 Random Forest - DevignX (ULTIMATE FIX)
# ================================

import os
import json
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

# ================================
# 🔹 1. LOAD DATA (ROBUST)
# ================================
data_path = "/kaggle/input/datasets/nikunjnawal009/randomf-devignx"
file_path = os.path.join(data_path, os.listdir(data_path)[0])

print("Using file:", file_path)

# Try JSONL
data = []
try:
    with open(file_path, "r") as f:
        for line in f:
            data.append(json.loads(line))
    df = pd.DataFrame(data)
    print("Loaded as JSONL")
except:
    pass

# If empty → try full JSON
if len(data) == 0:
    try:
        with open(file_path, "r") as f:
            data = json.load(f)
        df = pd.DataFrame(data)
        print("Loaded as JSON array")
    except:
        pass

# If still empty → try CSV
if 'df' not in locals() or df.shape[0] == 0:
    df = pd.read_csv(file_path)
    print("Loaded as CSV")

print("\nColumns:", df.columns)
print("Shape:", df.shape)

# ================================
# 🔹 2. SAFE COLUMN DETECTION
# ================================
code_col = None
label_col = None

for col in df.columns:
    if any(k in col.lower() for k in ["func", "code", "source", "snippet"]):
        code_col = col
    if any(k in col.lower() for k in ["target", "label", "vul", "bug"]):
        label_col = col

# 🔥 HARD FIX (Devign standard)
if code_col is None and "func" in df.columns:
    code_col = "func"
if label_col is None and "target" in df.columns:
    label_col = "target"

# FINAL fallback
if code_col is None or label_col is None:
    print("\n⚠️ Could not auto-detect columns. Using manual fallback.")
    code_col = df.columns[0]
    label_col = df.columns[-1]

print(f"\n✅ Code column: {code_col}")
print(f"✅ Label column: {label_col}")

# ================================
# 🔹 3. CLEAN DATA
# ================================
df = df[[code_col, label_col]]

df[code_col] = df[code_col].fillna("").astype(str)

df[label_col] = pd.to_numeric(df[label_col], errors="coerce")
df = df.dropna(subset=[label_col])
df[label_col] = df[label_col].astype(int)

print("Final dataset shape:", df.shape)

# ================================
# 🔹 4. SPLIT
# ================================
X = df[code_col]
y = df[label_col]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.1, random_state=42
)

# ================================
# 🔹 5. TF-IDF
# ================================
vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1,2))

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

# ================================
# 🔹 6. RANDOM FOREST
# ================================
model = RandomForestClassifier(
    n_estimators=150,
    max_depth=25,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train_tfidf, y_train)

# ================================
# 🔹 7. RESULTS
# ================================
y_pred = model.predict(X_test_tfidf)

print("\n📊 RESULTS")
print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, zero_division=0))
print("Recall   :", recall_score(y_test, y_pred, zero_division=0))
print("F1 Score :", f1_score(y_test, y_pred, zero_division=0))

print("\n🔍 Classification Report:\n")
print(classification_report(y_test, y_pred, zero_division=0))

Using file: /kaggle/input/datasets/nikunjnawal009/randomf-devignx/Devignx_validation.csv
Loaded as CSV

Columns: Index(['code', 'label'], dtype='object')
Shape: (2732, 2)

✅ Code column: code
✅ Label column: label
Final dataset shape: (2732, 2)

📊 RESULTS
Accuracy : 0.5620437956204379
Precision: 0.4878048780487805
Recall   : 0.16806722689075632
F1 Score : 0.25

🔍 Classification Report:

              precision    recall  f1-score   support

           0       0.58      0.86      0.69       155
           1       0.49      0.17      0.25       119

    accuracy                           0.56       274
   macro avg       0.53      0.52      0.47       274
weighted avg       0.54      0.56      0.50       274

